# Entanglement and marginals — the whole is pure, the parts are not

**The punchline.** A Bell pair is in a perfectly definite state: one state vector, no
uncertainty, nothing unknown about it. Ask about *either half on its own* and the answer
is the maximally mixed state — total ignorance, the worst any qubit can do. Nothing was
lost in between. Knowing everything about the pair is compatible with knowing nothing
about the parts, because the pair is not made of parts.

Background: **[02 — Entanglement](../02-entanglement.ipynb)** builds the partial trace,
the density matrix and the Schmidt decomposition from scratch. This exhibit assumes it,
and pushes on one consequence.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qsim import Circuit, viz
from qsim.gates import CNOT, H, Ry

np.set_printoptions(precision=3, suppress=True)

## 1. Two qubits, two gates

$H$ on the first qubit, then `CNOT` from the first to the second. The result is the Bell
state $(\lvert 00\rangle + \lvert 11\rangle)/\sqrt2$.

The bar chart below is the whole state: bar height is the size of an amplitude, bar
colour is its phase. Two bars of equal height, same colour, and two basis states absent
entirely.

In [ ]:
bell = Circuit(name="bell", seed=1234)
a, b = bell.alloc_many(2)
H(a)
CNOT(a, b)

print("the state:", bell.inspect.ket())
print("norm     :", bell.inspect.norm())
print("entropy of the whole two-qubit system:",
      bell.inspect.entanglement_entropy([a, b]), "bits")

fig_bell = viz.amplitudes(bell)

Entropy zero. The state is **pure** — a single state vector, not a probability
distribution over several. There is nothing about this pair that we failed to write down.

Now ask about qubit `a` by itself.

In [ ]:
rho_a = bell.inspect.reduced_density_matrix([a])

print("reduced density matrix of a:")
print(rho_a)
print()
print("equal to I/2 (the maximally mixed state)?",
      np.allclose(rho_a, np.eye(2) / 2))
print("Bloch vector of a          :", np.round(bell.inspect.bloch_vector(a), 12))
print("entanglement entropy of a  :", round(bell.inspect.entanglement_entropy([a]), 12),
      "bits  (1 is the most a qubit can have)")

$\rho_a = I/2$ exactly. Every diagonal entry $1/2$: both outcomes equally likely. Every
off-diagonal entry $0$: no coherence, so there is no measurement axis — none, in any
direction — along which this qubit gives a predictable answer. The Bloch vector is
$(0,0,0)$: not a point on the sphere at all, but its centre.

That is what "no state of its own" means, made numerical. Qubit `a` is not in some state
we merely failed to compute. The description of the pair is complete, and it assigns `a`
no state, because the information that would have been `a`'s is stored in the
*correlation* instead.

## 2. Contrast: a product state with the same local statistics

Replace the `CNOT` with a second $H$. Now each qubit is independently in
$\lvert +\rangle$, and a computational-basis measurement of `a` is still a fair coin —
$P(0) = 1/2$, exactly as in the Bell state.

In [ ]:
product = Circuit(name="product", seed=1234)
c, d = product.alloc_many(2)
H(c)
H(d)

for label, qc, q in [("Bell", bell, a), ("product H(c) H(d)", product, c)]:
    rho = qc.inspect.reduced_density_matrix([q])
    p0 = float(np.real(rho[0, 0]))
    x, y, z = qc.inspect.bloch_vector(q)
    length = float(np.sqrt(x * x + y * y + z * z))
    print(f"{label:>18}:  P(0) = {p0:.3f}   Bloch = ({x:+.3f}, {y:+.3f}, {z:+.3f})"
          f"   length {length:.3f}   S = {qc.inspect.entanglement_entropy([q]):.3f} bits")

Identical in the first column, opposite in the last. Drawn inside the sphere, the two
qubits could hardly look less alike — one arrow reaching the surface, one arrow that is
not an arrow at all.

In [ ]:
fig_bell_bloch = viz.bloch(a)
fig_product_bloch = viz.bloch(c)

Same $P(0)$, completely different qubits. The product state's `c` sits on the surface of
the sphere at $(1,0,0)$ — it *is* $\lvert +\rangle$, a definite pure state, and measuring
it along $x$ gives 0 with certainty. The Bell state's `a` is an arrow of length zero.

The length of the Bloch vector is the honest summary: **1 for a qubit with a state of its
own, 0 for one maximally entangled with something else, and in between for in between.**
Section 4 turns that "in between" into a dial.

## 3. Three qubits: measuring one of them

The GHZ state $(\lvert 000\rangle + \lvert 111\rangle)/\sqrt2$ is the same trick with one
more `CNOT`. Measure the first qubit and the other two stop being a superposition at all
— not because anything travelled to them, but because the branch in which they were
anything else has been discarded.

The measurement is seeded, so the outcome below is reproducible; run it with a different
seed and you get the other branch, with the rest of the story unchanged.

In [ ]:
ghz = Circuit(name="ghz", seed=7)
g0, g1, g2 = ghz.alloc_many(3)
H(g0)
CNOT(g0, g1)
CNOT(g1, g2)

print("before:", ghz.inspect.ket())
print("  entropy of g0 alone      :", round(ghz.inspect.entanglement_entropy([g0]), 6), "bits")
print("  entropy of {g1, g2}      :", round(ghz.inspect.entanglement_entropy([g1, g2]), 6), "bits")
print("  entropy of g1 alone      :", round(ghz.inspect.entanglement_entropy([g1]), 6), "bits")

outcome = ghz.measure(g0)

print()
print(f"measured g0 -> {outcome}")
print("after :", ghz.inspect.ket())
print("  entropy of {g1, g2}      :", round(ghz.inspect.entanglement_entropy([g1, g2]), 6), "bits")
print("  Bloch vector of g1       :", np.round(ghz.inspect.bloch_vector(g1), 6))

Before the measurement, `g1` had a Bloch vector of length 0 and one full bit of
entanglement entropy. After it, `g1` sits at a pole of the sphere with entropy 0 — a
perfectly definite state, and the *same* definite state as `g2`.

Note which entropies changed. `{g1, g2}` went from 1 bit (entangled with `g0`) to 0
(a product state of their own). But the two of them were, and remain, perfectly
correlated with each other. Measurement did not destroy the correlation; it converted a
quantum one into a classical one.

## 4. Entanglement as a continuous dial

Nothing forces entanglement to be all-or-nothing. Replace the $H$ with a rotation of
adjustable angle:

$$R_y(\theta)\lvert 0\rangle = \cos(\theta/2)\lvert 0\rangle + \sin(\theta/2)\lvert 1\rangle,
\qquad \text{then } \mathrm{CNOT} \Rightarrow
\cos(\tfrac{\theta}{2})\lvert 00\rangle + \sin(\tfrac{\theta}{2})\lvert 11\rangle.$$

At $\theta = 0$ that is $\lvert 00\rangle$: a product state, no entanglement. At
$\theta = \pi/2$ it is the Bell state. In between, the entropy of one qubit should be the
**binary entropy** of $p = \cos^2(\theta/2)$ — the classical Shannon entropy of a coin
with bias $p$, which is what the pair of Schmidt coefficients amounts to here.

In [ ]:
def binary_entropy(p: np.ndarray) -> np.ndarray:
    """Shannon entropy of a biased coin, in bits: -p log2 p - (1-p) log2 (1-p).

    np.where evaluates *both* branches before selecting, so log2(0) would still be
    computed (and warn) at p = 0 or 1. Clipping the input keeps the arithmetic finite;
    the clipped values are then discarded by the where, so the answer is unaffected.
    """
    safe = np.clip(p, 1e-300, 1.0 - 1e-16)
    terms = -safe * np.log2(safe) - (1.0 - safe) * np.log2(1.0 - safe)
    return np.where((p <= 0.0) | (p >= 1.0), 0.0, terms)


thetas = np.linspace(0.0, np.pi, 121)
entropies = []
lengths = []

for theta in thetas:
    qc = Circuit(name="dial", seed=1234)
    x0, x1 = qc.alloc_many(2)
    Ry(x0, theta=theta)
    CNOT(x0, x1)
    entropies.append(qc.inspect.entanglement_entropy([x0]))
    bx, by, bz = qc.inspect.bloch_vector(x0)
    lengths.append(np.sqrt(bx * bx + by * by + bz * bz))

fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.plot(thetas, entropies, lw=2, color="crimson", label="entanglement entropy of x0")
ax.plot(thetas, binary_entropy(np.cos(thetas / 2) ** 2), "k--", lw=1.2,
        label=r"binary entropy of $\cos^2(\theta/2)$")
ax.plot(thetas, lengths, lw=2, color="teal", label="length of x0's Bloch vector")
ax.set_xticks([0, np.pi / 2, np.pi], ["0", "π/2", "π"])
ax.set_xlabel(r"$\theta$ in $R_y(\theta)$")
ax.set_ylim(-0.05, 1.4)
ax.legend(fontsize=8, loc="upper center", ncol=2)
ax.set_title("entanglement is a dial, and the Bloch vector shrinks as it turns")
fig.tight_layout()

The two curves move in opposite directions and they are the same fact told twice. As the
entropy of one qubit rises from 0 to 1 bit, the length of its Bloch vector falls from 1
to 0. Entanglement with the outside is *exactly* the amount by which a qubit stops having
a state of its own.

At $\theta = \pi$ the entropy drops back to zero: $R_y(\pi)\lvert 0\rangle = \lvert
1\rangle$, so the state is $\lvert 11\rangle$ — a product state again. Entanglement is
about the *superposition* being shared, not about the `CNOT` having been applied.

## Where to go next

- **[03 — Bell tests and teleportation](../03-bell-tests-teleportation.ipynb)**: why this
  correlation cannot be explained by the two qubits having agreed in advance.
- **[decoherence_dial](decoherence_dial.ipynb)**: the same shrinking Bloch vector, but
  with the partner qubit relabelled "the environment" — which is all decoherence is.
- **[wigners_friend](wigners_friend.ipynb)**: the measurement in section 3, modelled as
  an entangling gate instead of an act of God.

## Assertions

The claims above, re-checked numerically.

In [ ]:
# 1. The Bell pair is pure as a whole and maximally mixed in each part.
assert np.isclose(bell.inspect.entanglement_entropy([a, b]), 0.0, atol=1e-12)
assert np.allclose(bell.inspect.reduced_density_matrix([a]), np.eye(2) / 2)
assert np.allclose(bell.inspect.reduced_density_matrix([b]), np.eye(2) / 2)
assert np.allclose(bell.inspect.bloch_vector(a), (0.0, 0.0, 0.0), atol=1e-12)
assert np.isclose(bell.inspect.entanglement_entropy([a]), 1.0)

# 2. The product state has the same P(0) locally and a full-length Bloch vector.
assert np.isclose(bell.inspect.reduced_density_matrix([a])[0, 0].real,
                  product.inspect.reduced_density_matrix([c])[0, 0].real)
assert np.allclose(product.inspect.bloch_vector(c), (1.0, 0.0, 0.0))
assert np.isclose(product.inspect.entanglement_entropy([c]), 0.0, atol=1e-12)

# 3. Measuring one qubit of GHZ leaves the other two unentangled, definite and equal.
assert np.isclose(ghz.inspect.entanglement_entropy([g1, g2]), 0.0, atol=1e-12)
assert np.isclose(ghz.inspect.entanglement_entropy([g1]), 0.0, atol=1e-12)
assert ghz.measure(g1) == outcome and ghz.measure(g2) == outcome

# 4. The dial follows the binary entropy of cos^2(theta/2), and Bloch length = sqrt(1-...)
assert np.allclose(entropies, binary_entropy(np.cos(thetas / 2) ** 2), atol=1e-9)
assert np.allclose(lengths, np.abs(np.cos(thetas)), atol=1e-9)
assert np.isclose(entropies[0], 0.0, atol=1e-12)
assert np.isclose(entropies[-1], 0.0, atol=1e-12)
assert np.isclose(max(entropies), 1.0)

print("all assertions passed")